<a href="https://colab.research.google.com/github/dataguirre/curso-ia-ciencia-de-datos/blob/main/entregables/01-herramienta-rag-pdf-gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pregúntale a tus PDF: herramienta OCR + RAG con Gradio

Notebook **independiente** que convierte el Workshop 5 en una herramienta: subes uno o varios PDF (o un
`.zip` con PDF), la app detecta página por página si hay texto o si hay que leerlo con OCR, limpia
encabezados y pies, arma el índice y te deja hacer preguntas citando documento y página.

```text
PDF / .zip ─┬─ página con capa de texto ─────────────────────┐
            └─ página sin texto → imagen → (preproceso) → OCR ┴→ limpieza → chunks con página → FAISS
                                                                                                  │
                                    pregunta → embedding → top-k fragmentos → Groq → respuesta + fuentes
```

La interfaz tiene tres pestañas:

1. **Cargar documentos:** procesa los PDF y muestra qué se leyó con texto nativo y qué con OCR, con la
   confianza del OCR y las páginas que conviene revisar. Descarga un CSV con el texto de cada página.
2. **Preguntar:** chat con RAG. Cada respuesta muestra los fragmentos usados y su similitud.
3. **Revisar páginas:** compara la imagen de una página con el texto que se extrajo de ella.

> **No hace falta GPU.** Sí necesitas tu `GROQ_API_KEY` en *Secrets* (la llave del panel izquierdo), como
> en los workshops.

**Cómo usarlo:** `Entorno de ejecución > Ejecutar todas`. La última celda imprime un enlace
`https://….gradio.live`; ábrelo y sube tus PDF ahí.

## 1. Instalación e imports

In [ ]:
!apt-get -qq install -y tesseract-ocr tesseract-ocr-spa poppler-utils > /dev/null
!pip install -q pytesseract pdf2image pdfplumber sentence-transformers faiss-cpu groq gradio

import inspect
import os
import re
import shutil
import tempfile
import time
import zipfile
from collections import Counter
from pathlib import Path

import cv2
import faiss
import gradio as gr
import numpy as np
import pandas as pd
import pdfplumber
import pytesseract
from PIL import Image
from pdf2image import convert_from_path

print("✓ Tesseract", pytesseract.get_tesseract_version())
print("✓ Idiomas:", pytesseract.get_languages())
print("✓ Gradio", gr.__version__)

✓ Tesseract 5.3.4
✓ Idiomas: ['eng', 'osd', 'spa']
✓ Gradio 6.26.0


## 2. Configuración

Lo único que normalmente hay que tocar está en esta celda. Los valores por defecto salen de lo que medimos
en el workshop.

In [ ]:
# ---- Modelos ----
MODELO_EMBEDDING = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MODELO_GROQ      = "openai/gpt-oss-20b"

# ---- Lectura de los PDF ----
MIN_CARACTERES_PAGINA = 25     # una página con menos texto que esto se trata como escaneada
DPI_OCR               = 300    # Tesseract rinde mejor alrededor de 300 dpi
IDIOMA_OCR            = "spa"
PREPROCESADO_DEFECTO  = "sin preprocesar"   # en el workshop, preprocesar empeoró los escaneos leves
REINTENTO_OCR         = "corregir iluminación"  # si el OCR sale casi vacío, reintenta con esto (None = no)
CONFIANZA_MINIMA      = 70     # confianza media del OCR (0-100) bajo la cual una página se marca para revisar
MAX_PAGINAS_POR_PDF   = 300    # protección contra archivos enormes

# ---- Limpieza y RAG ----
UMBRAL_REPETIDAS = 0.5   # línea presente en ≥50 % de las páginas → encabezado o pie
TAMANO_CHUNK     = 500
K                = 3

# ---- Archivos de trabajo ----
CARPETA_TRABAJO = Path("pdfs_subidos")
CSV_TEXTO       = "texto_extraido.csv"

NO_ENCONTRADO = "No encontré esa información en los documentos."
SYSTEM_PROMPT = (
    "Eres un asistente que responde preguntas sobre los documentos que subió el usuario. Responde en "
    "español, en una a tres frases, usando únicamente los fragmentos entregados, y termina citando la "
    "fuente entre paréntesis, así: (documento, página N). Si una circular o disposición temporal aplica a "
    "la fecha de la pregunta, prevalece sobre las políticas generales. Si la respuesta no está en los "
    f"fragmentos, responde exactamente: {NO_ENCONTRADO}"
)

CARPETA_TRABAJO.mkdir(exist_ok=True)
print("✓ Configuración lista")

✓ Configuración lista


## 3. Modelos: embeddings y Groq

El modelo de embeddings (unos 470 MB) se descarga la primera vez. La llave de Groq se lee de *Secrets*; si
corres el notebook fuera de Colab, se lee de la variable de entorno `GROQ_API_KEY`.

In [ ]:
from groq import Groq, RateLimitError
from sentence_transformers import SentenceTransformer


def leer_groq_api_key() -> str:
    """Busca la llave en Secrets de Colab y, si no está, en las variables de entorno."""
    try:
        from google.colab import userdata
        llave = userdata.get("GROQ_API_KEY")
        if llave:
            return llave
    except Exception:
        pass
    llave = os.environ.get("GROQ_API_KEY")
    if not llave:
        raise RuntimeError(
            "No encontré GROQ_API_KEY. En Colab agrégala en Secrets (icono de llave, panel izquierdo) "
            "y activa 'Acceso del notebook'."
        )
    return llave


embedder = SentenceTransformer(MODELO_EMBEDDING)
client = Groq(api_key=leer_groq_api_key())

print(f"✓ {MODELO_EMBEDDING} cargado")
print(f"✓ Cliente de Groq listo ({MODELO_GROQ})")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 cargado
✓ Cliente de Groq listo (openai/gpt-oss-20b)


## 4. Leer un PDF, página por página

Estas funciones vienen de las Actividades 1 a 3, con dos cambios pensados para documentos reales:

- **La detección es por página, no por documento.** Un PDF puede ser *mixto*: un contrato generado en Word
  con un anexo escaneado al final. Si una página trae texto, se extrae gratis; si no, solo esa página pasa
  por OCR.
- **Si el OCR sale casi vacío, se reintenta una vez** con `corregir iluminación`, y se queda con el
  resultado que tenga más texto. Es el caso del nivel `fuerte` de la Actividad 3: con sombra, Tesseract puede
  devolver una página en blanco. El reintento solo se dispara cuando falla, así que no empeora las páginas
  que ya se leían bien.
- **El OCR devuelve también su confianza.** Usamos `image_to_data` en lugar de `image_to_string` (la
  extensión "confianza por palabra" del workshop). Así la app puede señalar qué páginas conviene revisar,
  igual que la columna `requiere_revision` del pipeline de tiquetes. No reemplaza al CER (para eso hace
  falta la referencia nativa), pero sirve de alarma cuando no la tenemos.

In [ ]:
# --- Preprocesamiento (Actividad 3) ---

def binarizar_fijo(imagen: Image.Image, umbral: int = 150) -> Image.Image:
    """Binariza una imagen con un umbral fijo."""
    gris = np.asarray(imagen.convert("L"))
    return Image.fromarray(np.where(gris > umbral, 255, 0).astype(np.uint8))


def binarizar_otsu(imagen: Image.Image) -> Image.Image:
    """Binariza una imagen eligiendo el umbral con el método de Otsu."""
    gris = np.asarray(imagen.convert("L"))
    _, binaria = cv2.threshold(gris, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return Image.fromarray(binaria)


def corregir_iluminacion(imagen: Image.Image, tamano_fondo: int = 51) -> Image.Image:
    """Corrige la iluminación desigual y binariza con Otsu."""
    gris = np.asarray(imagen.convert("L"))
    fondo = cv2.medianBlur(gris, tamano_fondo)
    corregida = cv2.divide(gris, fondo, scale=255)
    _, binaria = cv2.threshold(corregida, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return Image.fromarray(binaria)


PREPROCESADOS = {
    "sin preprocesar": None,
    "umbral fijo": binarizar_fijo,
    "otsu": binarizar_otsu,
    "corregir iluminación": corregir_iluminacion,
}


def aplicar_preprocesado(imagen: Image.Image, nombre: str) -> Image.Image:
    """Aplica el método de PREPROCESADOS con ese nombre (o ninguno)."""
    funcion = PREPROCESADOS[nombre]
    return imagen if funcion is None else funcion(imagen)


# --- OCR con confianza ---

def ocr_imagen(imagen: Image.Image, idioma: str = IDIOMA_OCR) -> tuple[str, float | None]:
    """Aplica OCR a una imagen y devuelve (texto, confianza media de las palabras entre 0 y 100)."""
    datos = pytesseract.image_to_data(imagen, lang=idioma, output_type=pytesseract.Output.DICT)

    lineas, confianzas = {}, []
    for i, palabra in enumerate(datos["text"]):
        palabra = palabra.strip()
        if not palabra:
            continue
        clave = (datos["block_num"][i], datos["par_num"][i], datos["line_num"][i])
        lineas.setdefault(clave, []).append(palabra)
        confianza = float(datos["conf"][i])
        if confianza >= 0:
            confianzas.append(confianza)

    # Reconstruye el texto: una línea por renglón y una línea en blanco entre párrafos
    partes, parrafo_anterior = [], None
    for (bloque, parrafo, _), palabras in lineas.items():
        if parrafo_anterior is not None and (bloque, parrafo) != parrafo_anterior:
            partes.append("")
        partes.append(" ".join(palabras))
        parrafo_anterior = (bloque, parrafo)

    return "\n".join(partes), (float(np.mean(confianzas)) if confianzas else None)


# --- Lectura página por página (Actividades 1 y 2) ---

def leer_pdf(ruta, forzar_ocr: bool = False, preprocesado: str = PREPROCESADO_DEFECTO,
             dpi: int = DPI_OCR, al_avanzar=None) -> list[dict]:
    """Lee un PDF y devuelve un diccionario por página con su texto y cómo se obtuvo.

    Args:
        ruta: Ruta al PDF.
        forzar_ocr: Si es True, aplica OCR aunque la página tenga capa de texto.
        preprocesado: Nombre de un método de PREPROCESADOS.
        dpi: Resolución para convertir las páginas escaneadas en imagen.
        al_avanzar: Función opcional al_avanzar(pagina, total), para mostrar el progreso.

    Returns:
        Lista de {"pagina", "metodo", "preprocesado", "texto", "confianza"}; "preprocesado" y
        "confianza" son None en las páginas nativas.
    """
    paginas = []
    with pdfplumber.open(ruta) as pdf:
        total = len(pdf.pages)
        if total > MAX_PAGINAS_POR_PDF:
            raise ValueError(f"tiene {total} páginas; el máximo es {MAX_PAGINAS_POR_PDF}")

        for numero, pagina in enumerate(pdf.pages, start=1):
            texto = pagina.extract_text() or ""
            if not forzar_ocr and len(texto.strip()) >= MIN_CARACTERES_PAGINA:
                paginas.append({"pagina": numero, "metodo": "texto nativo", "preprocesado": None,
                                "texto": texto, "confianza": None})
            else:
                imagen = convert_from_path(ruta, dpi=dpi, first_page=numero, last_page=numero)[0]
                usado = preprocesado
                texto, confianza = ocr_imagen(aplicar_preprocesado(imagen, usado))

                if (len(texto.strip()) < MIN_CARACTERES_PAGINA
                        and REINTENTO_OCR and REINTENTO_OCR != preprocesado):
                    texto_2, confianza_2 = ocr_imagen(aplicar_preprocesado(imagen, REINTENTO_OCR))
                    if len(texto_2.strip()) > len(texto.strip()):
                        texto, confianza, usado = texto_2, confianza_2, REINTENTO_OCR

                paginas.append({"pagina": numero, "metodo": "OCR", "preprocesado": usado,
                                "texto": texto, "confianza": confianza})
            if al_avanzar:
                al_avanzar(numero, total)
    return paginas


def tipo_documento(paginas: list[dict]) -> str:
    """Clasifica un documento como nativo, escaneado o mixto según cómo se leyeron sus páginas."""
    metodos = {p["metodo"] for p in paginas}
    if metodos == {"texto nativo"}:
        return "nativo"
    if metodos == {"OCR"}:
        return "escaneado"
    return "mixto" if metodos else "vacío"


def requiere_revision(pagina: dict) -> bool:
    """Una página de OCR se marca si quedó vacía o si su confianza media es baja."""
    if pagina["metodo"] != "OCR":
        return False
    return not pagina["texto"].strip() or (pagina["confianza"] or 0) < CONFIANZA_MINIMA


# --- Limpieza de encabezados y pies (Actividad 1, Tarea 3) ---

PATRON_PAGINA = re.compile(r"Página \d+ de \d+")


def limpiar_paginas(paginas: list[str], umbral: float = UMBRAL_REPETIDAS) -> list[str]:
    """Elimina numeración, encabezados y pies de página repetidos."""
    lineas_por_pagina = [
        [PATRON_PAGINA.sub("", linea).strip() for linea in pagina.splitlines()]
        for pagina in paginas
    ]
    repetidas = set()
    if len(paginas) >= 3:
        conteo = Counter(linea for lineas in lineas_por_pagina for linea in set(lineas) if linea)
        repetidas = {linea for linea, n in conteo.items() if n / len(paginas) >= umbral}
    return [
        "\n".join(linea for linea in lineas if linea and linea not in repetidas)
        for lineas in lineas_por_pagina
    ]


print("✓ Funciones de lectura listas")

✓ Funciones de lectura listas


## 5. Índice y RAG

Las piezas del Workshop 4 y de la Actividad 4: cada chunk recuerda su documento y su página, así que la
respuesta puede citar la fuente.

In [ ]:
def dividir_documento(documento: str, tamano: int = TAMANO_CHUNK) -> list[str]:
    """Divide un texto en fragmentos de aproximadamente tamano caracteres."""
    if tamano <= 0:
        raise ValueError("tamano debe ser positivo")
    return [
        documento[i:i + tamano].strip()
        for i in range(0, len(documento), tamano)
        if documento[i:i + tamano].strip()
    ]


def crear_embeddings(textos: list[str]) -> np.ndarray:
    """Convierte textos en embeddings normalizados."""
    return embedder.encode(textos, convert_to_numpy=True, normalize_embeddings=True).astype("float32")


def construir_indice(embeddings: np.ndarray) -> faiss.IndexFlatIP:
    """Construye un índice FAISS para similitud coseno."""
    indice = faiss.IndexFlatIP(embeddings.shape[1])
    indice.add(embeddings)
    return indice


def crear_chunks(corpus: dict, tamano: int = TAMANO_CHUNK) -> list[dict]:
    """Divide cada página del corpus {documento: páginas} en chunks con documento y página."""
    return [
        {"texto": fragmento, "documento": doc, "pagina": numero}
        for doc, paginas in corpus.items()
        for numero, pagina in enumerate(paginas, start=1)
        for fragmento in dividir_documento(pagina, tamano)
    ]


def buscar(pregunta: str, chunks: list[dict], indice, k: int = K) -> list[tuple[dict, float]]:
    """Recupera los k chunks más similares a la pregunta, con su similitud."""
    similitudes, posiciones = indice.search(crear_embeddings([pregunta]), min(k, len(chunks)))
    return [(chunks[p], float(s)) for s, p in zip(similitudes[0], posiciones[0]) if p >= 0]


def preguntar_groq(prompt: str, system_prompt: str = None, modelo: str = MODELO_GROQ,
                   temperature: float = 0.0, reintentos: int = 5) -> str:
    """Envía un prompt a Groq y reintenta si se alcanza el límite de uso."""
    mensajes = []
    if system_prompt:
        mensajes.append({"role": "system", "content": system_prompt})
    mensajes.append({"role": "user", "content": prompt})

    for intento in range(reintentos):
        try:
            respuesta = client.chat.completions.create(
                model=modelo, messages=mensajes, temperature=temperature, reasoning_effort="low",
            )
            return respuesta.choices[0].message.content
        except RateLimitError:
            time.sleep(10 * (intento + 1))
    raise RuntimeError("Groq sigue rechazando las solicitudes; espera un minuto y vuelve a intentar.")


def responder_con_rag(pregunta: str, base: dict, k: int = K) -> tuple[str, list]:
    """Responde con RAG sobre una base ya construida y devuelve también los fragmentos usados."""
    resultados = buscar(pregunta, base["chunks"], base["indice"], k=k)
    contexto = "\n\n".join(
        f"[Fragmento {i} | {c['documento']}, página {c['pagina']}]\n{c['texto']}"
        for i, (c, _) in enumerate(resultados, start=1)
    )
    respuesta = preguntar_groq(f"Fragmentos:\n\n{contexto}\n\nPregunta: {pregunta}",
                               system_prompt=SYSTEM_PROMPT)
    return respuesta, resultados


print("✓ Funciones de RAG listas")

✓ Funciones de RAG listas


## 6. Del archivo subido a la base de conocimiento

`preparar_entrada` acepta PDF sueltos y `.zip` (con los PDF sueltos o dentro de carpetas). `procesar_pdfs`
lee cada documento, arma el índice y escribe el CSV con el texto de cada página. Si un archivo falla (está
dañado o protegido con contraseña), se reporta en la tabla y los demás siguen.

In [ ]:
def preparar_entrada(rutas: list, carpeta: Path) -> list[Path]:
    """Copia los PDF (y el contenido de los .zip) a una carpeta de trabajo y devuelve sus rutas."""
    carpeta.mkdir(parents=True, exist_ok=True)
    encontrados = []
    for ruta in map(Path, rutas):
        if ruta.suffix.lower() == ".zip":
            if not zipfile.is_zipfile(ruta):
                raise ValueError(f"{ruta.name} no es un .zip válido (¿se subió completo?)")
            destino = carpeta / f"_zip_{ruta.stem}"
            with zipfile.ZipFile(ruta) as z:
                z.extractall(destino)
            encontrados += [
                p for p in destino.rglob("*")
                if p.suffix.lower() == ".pdf" and not p.name.startswith(".") and "__MACOSX" not in p.parts
            ]
        elif ruta.suffix.lower() == ".pdf":
            encontrados.append(ruta)

    # Copia con nombres únicos: dos "informe.pdf" en carpetas distintas no se pisan
    pdfs, usados = [], Counter()
    for ruta in sorted(encontrados, key=lambda p: p.name.lower()):
        usados[ruta.stem] += 1
        nombre = ruta.stem if usados[ruta.stem] == 1 else f"{ruta.stem}_{usados[ruta.stem]}"
        destino = carpeta / f"{nombre}.pdf"
        if ruta.resolve() != destino.resolve():
            shutil.copy(ruta, destino)
        pdfs.append(destino)

    if not pdfs:
        raise FileNotFoundError("No encontré ningún PDF en lo que subiste.")
    return pdfs


def procesar_pdfs(rutas: list, forzar_ocr: bool = False, preprocesado: str = PREPROCESADO_DEFECTO,
                  al_avanzar=None) -> tuple[dict, pd.DataFrame, pd.DataFrame]:
    """Lee todos los PDF, construye la base del RAG y escribe el CSV con el texto por página.

    Args:
        rutas: PDF y/o .zip a procesar.
        forzar_ocr: Aplica OCR a todas las páginas.
        preprocesado: Nombre de un método de PREPROCESADOS.
        al_avanzar: Función opcional al_avanzar(fraccion, mensaje) para mostrar el progreso.

    Returns:
        (base, resumen por documento, detalle por página). base["csv"] es la ruta del CSV.
    """
    carpeta = Path(tempfile.mkdtemp(dir=CARPETA_TRABAJO))
    pdfs = preparar_entrada(rutas, carpeta)

    documentos, resumen = {}, []
    for i, ruta in enumerate(pdfs):
        doc = ruta.stem
        inicio = time.time()

        def avance(pagina, total, i=i, doc=doc):
            if al_avanzar:
                al_avanzar((i + pagina / total) / len(pdfs), f"{doc}: página {pagina} de {total}")

        try:
            paginas = leer_pdf(ruta, forzar_ocr=forzar_ocr, preprocesado=preprocesado, al_avanzar=avance)
        except Exception as e:
            resumen.append({"documento": doc, "tipo": "error", "error": f"{type(e).__name__}: {e}"})
            continue

        limpias = limpiar_paginas([p["texto"] for p in paginas])
        for pagina, limpia in zip(paginas, limpias):
            pagina["texto_limpio"] = limpia
            pagina["requiere_revision"] = requiere_revision(pagina)
        documentos[doc] = {"ruta": str(ruta), "paginas": paginas}

        confianzas = [p["confianza"] for p in paginas if p["confianza"] is not None]
        resumen.append({
            "documento": doc,
            "tipo": tipo_documento(paginas),
            "paginas": len(paginas),
            "paginas_ocr": sum(p["metodo"] == "OCR" for p in paginas),
            "confianza_ocr": round(float(np.mean(confianzas)), 1) if confianzas else None,
            "paginas_a_revisar": sum(p["requiere_revision"] for p in paginas),
            "caracteres": sum(len(p["texto_limpio"]) for p in paginas),
            "segundos": round(time.time() - inicio, 1),
            "error": "",
        })

    resumen_df = pd.DataFrame(resumen)
    # Enteros nullable: evita que una fila con error convierta la columna en 3.0
    for columna in ["paginas", "paginas_ocr", "paginas_a_revisar", "caracteres"]:
        if columna in resumen_df:
            resumen_df[columna] = pd.array(resumen_df[columna], dtype="Int64")
    detalle_df = pd.DataFrame([
        {"documento": doc, **{c: p[c] for c in ("pagina", "metodo", "preprocesado", "confianza", "requiere_revision")},
         "caracteres": len(p["texto_limpio"]), "texto_extraido": p["texto"], "texto_limpio": p["texto_limpio"]}
        for doc, d in documentos.items() for p in d["paginas"]
    ])
    csv_salida = carpeta / CSV_TEXTO   # un CSV por procesamiento: dos usuarios no se pisan
    detalle_df.to_csv(csv_salida, index=False, encoding="utf-8-sig")

    corpus = {doc: [p["texto_limpio"] for p in d["paginas"]] for doc, d in documentos.items()}
    chunks = crear_chunks(corpus)
    if not chunks:
        raise ValueError("No se pudo extraer texto de ningún documento.")
    if al_avanzar:
        al_avanzar(1.0, f"Creando embeddings de {len(chunks)} fragmentos")
    base = {
        "documentos": documentos,
        "chunks": chunks,
        "indice": construir_indice(crear_embeddings([c["texto"] for c in chunks])),
        "csv": str(csv_salida),
    }
    return base, resumen_df, detalle_df


print("✓ Funciones del pipeline listas")

✓ Funciones del pipeline listas


## 7. Prueba sin interfaz (opcional)

Antes de abrir la app conviene probar con un documento que conozcas y comparar con lo que está impreso. Sube
un PDF o un `.zip` al panel de archivos, pon su nombre en `PRUEBA` y escribe una pregunta cuya respuesta
sepas. Si dejas `PRUEBA` vacío, la celda no hace nada.

In [ ]:
PRUEBA          = ""    # por ejemplo "mis_pdfs.zip" o "contrato.pdf"
PREGUNTA_PRUEBA = "¿Cada cuántos kilómetros se hace el mantenimiento preventivo?"

if PRUEBA:
    base_prueba, resumen_prueba, detalle_prueba = procesar_pdfs(
        [PRUEBA], al_avanzar=lambda f, m: print(f"  {f:4.0%}  {m}")
    )
    display(resumen_prueba)
    respuesta, fuentes = responder_con_rag(PREGUNTA_PRUEBA, base_prueba)
    print(f"\nPregunta:  {PREGUNTA_PRUEBA}\nRespuesta: {respuesta}\n")
    for chunk, similitud in fuentes:
        print(f"  [{similitud:.2f}] {chunk['documento']}, página {chunk['pagina']}: {chunk['texto'][:80]}…")
else:
    print("Sin prueba: pon un archivo en PRUEBA para usarla.")

Sin prueba: pon un archivo en PRUEBA para usarla.


## 8. La interfaz

Cada persona que abre el enlace tiene su propia base (`gr.State`), así que dos usuarios con documentos
distintos no se mezclan.

In [ ]:
# Gradio 5 necesita type="messages" en el Chatbot; Gradio 6 ya lo usa por defecto y quitó el parámetro.
OPCIONES_CHAT = {"type": "messages"} if "type" in inspect.signature(gr.Chatbot.__init__).parameters else {}
COLUMNAS_FUENTES = ["#", "documento", "página", "similitud", "fragmento"]


def ui_procesar(archivos, preprocesado, forzar_ocr, progress=gr.Progress()):
    """Procesa lo subido y actualiza las tres pestañas."""
    if not archivos:
        raise gr.Error("Primero sube uno o varios PDF, o un .zip con PDF.")
    rutas = [a if isinstance(a, str) else a.name for a in archivos]
    try:
        base, resumen, _ = procesar_pdfs(
            rutas, forzar_ocr=forzar_ocr, preprocesado=preprocesado,
            al_avanzar=lambda fraccion, mensaje: progress(fraccion, desc=mensaje),
        )
    except (FileNotFoundError, ValueError) as e:
        raise gr.Error(str(e))

    listos = resumen[resumen["tipo"] != "error"]
    texto = (
        f"### ✓ {len(listos)} documento(s), {int(listos['paginas'].sum())} páginas, "
        f"{len(base['chunks'])} fragmentos indexados\n"
        f"- Leídas con texto nativo: **{int((listos['paginas'] - listos['paginas_ocr']).sum())}** páginas\n"
        f"- Leídas con OCR: **{int(listos['paginas_ocr'].sum())}** páginas\n"
        f"- Para revisar (confianza del OCR < {CONFIANZA_MINIMA}): **{int(listos['paginas_a_revisar'].sum())}**"
    )
    if (resumen["tipo"] == "error").any():
        texto += f"\n- ⚠️ No se pudieron leer: {', '.join(resumen.loc[resumen['tipo'] == 'error', 'documento'])}"
    texto += "\n\nYa puedes ir a la pestaña **2. Preguntar**."

    nombres = list(base["documentos"])
    primero = nombres[0]
    n_paginas = len(base["documentos"][primero]["paginas"])
    return (
        base, texto, resumen, base["csv"],
        gr.Dropdown(choices=nombres, value=primero),
        gr.Slider(minimum=1, maximum=max(n_paginas, 2), value=1, step=1, interactive=n_paginas > 1),
        [], pd.DataFrame(columns=COLUMNAS_FUENTES),
    )


def ui_preguntar(pregunta, historial, base, k):
    """Responde una pregunta con RAG y muestra los fragmentos usados."""
    historial = list(historial or [])
    pregunta = (pregunta or "").strip()
    if not pregunta:
        return "", historial, gr.skip()
    if base is None:
        raise gr.Error("Primero carga documentos en la pestaña 1.")

    try:
        respuesta, resultados = responder_con_rag(pregunta, base, k=int(k))
    except Exception as e:
        respuesta, resultados = f"⚠️ Error al consultar el modelo: {type(e).__name__}: {e}", []

    historial += [{"role": "user", "content": pregunta}, {"role": "assistant", "content": respuesta}]
    fuentes = pd.DataFrame(
        [[i, c["documento"], c["pagina"], round(s, 3), c["texto"]] for i, (c, s) in enumerate(resultados, 1)],
        columns=COLUMNAS_FUENTES,
    )
    return "", historial, fuentes


def ui_elegir_documento(base, documento):
    """Ajusta el deslizador de páginas al documento elegido."""
    if base is None or documento not in base["documentos"]:
        return gr.skip()
    n = len(base["documentos"][documento]["paginas"])
    return gr.Slider(minimum=1, maximum=max(n, 2), value=1, step=1, interactive=n > 1)


def ui_ver_pagina(base, documento, numero):
    """Muestra la imagen de una página junto al texto que se extrajo de ella."""
    if base is None or documento not in base["documentos"]:
        return None, "", "", ""
    datos = base["documentos"][documento]
    numero = min(max(int(numero), 1), len(datos["paginas"]))
    pagina = datos["paginas"][numero - 1]

    imagen = convert_from_path(datos["ruta"], dpi=100, first_page=numero, last_page=numero)[0]
    info = f"**{documento}**, página {numero} de {len(datos['paginas'])} · leída con **{pagina['metodo']}**"
    if pagina["preprocesado"] and pagina["preprocesado"] != "sin preprocesar":
        info += f" ({pagina['preprocesado']})"
    if pagina["confianza"] is not None:
        info += f" · confianza del OCR: **{pagina['confianza']:.0f}**"
    if pagina["requiere_revision"]:
        info += " · ⚠️ conviene revisarla"
    return imagen, info, pagina["texto"], pagina["texto_limpio"]


with gr.Blocks(title="Pregúntale a tus PDF") as demo:
    base_estado = gr.State(None)

    gr.Markdown(
        "# 📄 Pregúntale a tus PDF\n"
        "Sube PDF nativos o escaneados. La app detecta qué páginas necesitan OCR, limpia encabezados y pies, "
        "y responde tus preguntas citando documento y página."
    )

    with gr.Tab("1. Cargar documentos"):
        with gr.Row():
            with gr.Column(scale=1):
                entrada_archivos = gr.File(label="PDF o .zip con PDF", file_count="multiple",
                                           file_types=[".pdf", ".zip"], type="filepath")
                entrada_preprocesado = gr.Dropdown(
                    list(PREPROCESADOS), value=PREPROCESADO_DEFECTO, label="Preprocesamiento para OCR",
                    info="Solo afecta las páginas escaneadas. Prueba 'corregir iluminación' si hay sombras.",
                )
                entrada_forzar = gr.Checkbox(False, label="Forzar OCR en todas las páginas",
                                             info="Útil si sospechas que la capa de texto del PDF es mala.")
                boton_procesar = gr.Button("Procesar documentos", variant="primary")
            with gr.Column(scale=2):
                salida_resumen = gr.Markdown("Sube tus documentos y presiona **Procesar documentos**.")
                salida_tabla = gr.Dataframe(label="Resumen por documento", interactive=False, wrap=True)
                salida_csv = gr.File(label=f"Texto extraído por página ({CSV_TEXTO})")

    with gr.Tab("2. Preguntar"):
        chat = gr.Chatbot(label="Conversación", height=420, **OPCIONES_CHAT)
        with gr.Row():
            entrada_pregunta = gr.Textbox(label="Pregunta", scale=5,
                                          placeholder="Ej.: ¿Cuál es la velocidad máxima en temporada de lluvias?")
            entrada_k = gr.Slider(1, 8, value=K, step=1, label="Fragmentos (k)", scale=1)
        with gr.Row():
            boton_enviar = gr.Button("Preguntar", variant="primary")
            boton_limpiar = gr.Button("Borrar conversación")
        salida_fuentes = gr.Dataframe(label="Fragmentos usados en la última respuesta",
                                      headers=COLUMNAS_FUENTES, interactive=False, wrap=True)

    with gr.Tab("3. Revisar páginas"):
        with gr.Row():
            entrada_documento = gr.Dropdown(label="Documento", choices=[])
            entrada_pagina = gr.Slider(1, 2, value=1, step=1, label="Página", interactive=False)
        salida_info = gr.Markdown()
        with gr.Row():
            salida_imagen = gr.Image(label="Página original", type="pil", height=700)
            with gr.Column():
                salida_limpio = gr.Textbox(label="Texto limpio (lo que ve el RAG)", lines=15)
                with gr.Accordion("Texto extraído, antes de limpiar", open=False):
                    salida_crudo = gr.Textbox(show_label=False, lines=15)

    # --- Eventos ---
    boton_procesar.click(
        ui_procesar,
        inputs=[entrada_archivos, entrada_preprocesado, entrada_forzar],
        outputs=[base_estado, salida_resumen, salida_tabla, salida_csv,
                 entrada_documento, entrada_pagina, chat, salida_fuentes],
    ).then(
        ui_ver_pagina, inputs=[base_estado, entrada_documento, entrada_pagina],
        outputs=[salida_imagen, salida_info, salida_crudo, salida_limpio],
    )

    for disparador in (boton_enviar.click, entrada_pregunta.submit):
        disparador(ui_preguntar, inputs=[entrada_pregunta, chat, base_estado, entrada_k],
                   outputs=[entrada_pregunta, chat, salida_fuentes])
    boton_limpiar.click(lambda: ([], pd.DataFrame(columns=COLUMNAS_FUENTES)), outputs=[chat, salida_fuentes])

    entrada_documento.input(ui_elegir_documento, inputs=[base_estado, entrada_documento],
                            outputs=entrada_pagina).then(
        ui_ver_pagina, inputs=[base_estado, entrada_documento, entrada_pagina],
        outputs=[salida_imagen, salida_info, salida_crudo, salida_limpio],
    )
    entrada_pagina.release(ui_ver_pagina, inputs=[base_estado, entrada_documento, entrada_pagina],
                           outputs=[salida_imagen, salida_info, salida_crudo, salida_limpio])

print("✓ Interfaz lista")

✓ Interfaz lista


## 9. Lanzar la app

`share=True` crea un enlace público temporal (dura unas 72 horas o hasta que se cierre la sesión de Colab).
La celda queda corriendo mientras la app esté activa; para cambiar algo, detén la celda, edita y vuelve a
correr desde la sección 8.

In [ ]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://09725e05a3d8e96900.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Notas

**La detección es por página, pero sigue confiando en la capa de texto.** Si un PDF trae una capa de texto
generada por un OCR malo, la app la usa tal cual (pregunta 3 del workshop). Si ves texto raro en la pestaña
*Revisar páginas*, vuelve a procesar con **Forzar OCR**.

**El preprocesamiento es una hipótesis, no una receta.** Por eso el valor por defecto es `sin preprocesar`:
en el workshop cualquier binarización empeoró los escaneos leves, y solo `corregir iluminación` ayudó en los
que tenían sombra. La opción queda en la interfaz para probar con tus documentos.

**La confianza del OCR no es el CER.** Es lo que Tesseract cree de sí mismo, no una comparación contra la
verdad. Sirve para priorizar qué revisar, pero una página con confianza alta todavía puede traer un número
mal leído. Los datos que importan (teléfonos, valores, fechas) merecen una mirada en *Revisar páginas*.

**La limpieza de encabezados funciona peor en páginas de OCR.** El ruido cambia algún carácter del encabezado
en cada página y el conteo de líneas idénticas ya no lo reconoce (pregunta 3 de la Actividad 2). No rompe el
RAG, pero deja algo de ruido en los chunks. Una mejora posible es recortar la franja del encabezado y el pie
antes del OCR, o comparar líneas con distancia de edición.

**El OCR es la parte lenta.** Sin GPU, Tesseract tarda unos segundos por página a 300 dpi; un PDF escaneado
de 50 páginas puede tomar varios minutos. Las páginas nativas se leen casi al instante.

**Privacidad.** El texto de los fragmentos recuperados se envía a Groq para generar la respuesta, y el enlace
`gradio.live` es público mientras esté activo. No subas documentos confidenciales sin tenerlo en cuenta.

**Circulares y vigencia.** El prompt le pide al modelo que la circular temporal prevalezca, pero la solución
de fondo (pregunta 2 de la Actividad 4) sigue siendo guardar fechas de vigencia en los chunks y filtrar.